# 4 · Control flow

Two control-flow primitives:

1. **Conditional branching** via the `SKIPPED` sentinel — a node outputs `SKIPPED` on branches that shouldn't run. Skip propagates: any downstream node whose inputs are all `SKIPPED` is skipped too.
2. **For-each loops** via a compound node — a region of nodes between `for-each-start` and `for-each-end` that runs once per item.

Both primitives have ready-made nodes in the `conductor-nodes` sibling package (`logic.if_empty`, `loop.for_each_start` / `for_each_end`). This notebook imports them rather than hand-rolling.

In [ ]:
from typing import Annotated

import conductor_nodes
from conductor import GraphEdge, GraphNode, NodeRegistry, compile
from conductor.compound.for_each import FOR_EACH
from conductor.execution.engine import collect, execute
from conductor.widgets import Output, Text

registry = NodeRegistry()

# Bring in the standard-library nodes used below.
conductor_nodes.text.register(registry)       # text-uppercase, text-concat, ...
conductor_nodes.logic.register(registry)      # logic-if-empty, logic-if-equals, ...
conductor_nodes.loop.register(registry)       # for-each-start, for-each-end

print(f"Registered {len(registry.all())} nodes from conductor-nodes.")

We'll also register one small app-specific node to demonstrate mixing custom nodes with the standard library.

In [ ]:
@registry.node("prefix", version=1, name="Prefix", description="Adds prefix")
def prefix(text: Annotated[str, Text(label="Input")]) -> Annotated[str, Output(label="Output")]:
    return f">> {text}" 

## Conditional branching

`logic-if-empty` returns a 2-tuple — `(text, SKIPPED)` when text is non-empty, `(SKIPPED, text)` when empty. Wire downstream nodes to whichever branch they apply to; the dead branch's consumers are automatically skipped.

```text
  (input "hello") ──> if-empty ──(not empty)──> text-uppercase
                              └──(empty)──────> prefix    (skipped, because input was non-empty)
```

In [ ]:
nodes = [
    GraphNode("cond", "logic-if-empty@1", {"text": "hello"}),
    GraphNode("up", "text-uppercase@1", None),
    GraphNode("pf", "prefix@1", None),
]
edges = [
    GraphEdge("e1", "cond", "up", "output_1", "text"),   # "not empty" branch
    GraphEdge("e2", "cond", "pf", "output_2", "text"),   # "empty" branch
]

compiled = compile(nodes=nodes, edges=edges, registry=registry)
results = await collect(execute(compiled))

for nid, res in results.items():
    print(f"  {nid}: {res}")

Notice that `pf` does not appear in the results: it was skipped because its input was `SKIPPED`.

## For-each loop

`for-each-start` explodes a list into (item, index) per iteration; `for-each-end` collects each body's result back into a list. Pass `FOR_EACH` to `compile(compound_types=...)` so the compiler treats the region specially.

```text
  for-each-start(["alice", "bob", "charlie"])
      ─> text-uppercase   (loop body, runs 3 times)
      ─> for-each-end     ──> ["ALICE", "BOB", "CHARLIE"]
```

In [ ]:
nodes_b = [
    GraphNode("start", "for-each-start@1", {"items": ["alice", "bob", "charlie"]}),
    GraphNode("body", "text-uppercase@1", None),
    GraphNode("end", "for-each-end@1", None),
]
edges_b = [
    GraphEdge("e1", "start", "body", "output_1", "text"),
    GraphEdge("e2", "body", "end", "result", "item"),
]

compiled_b = compile(
    nodes=nodes_b,
    edges=edges_b,
    registry=registry,
    compound_types=[FOR_EACH],
)
results_b = await collect(execute(compiled_b))

for nid, res in results_b.items():
    print(f"  {nid}: {res}")

## Mixing in custom logic

The standard library's `logic-if-equals` routes on string equality. Combine it with your own nodes freely — the registry doesn't care where a node came from.

In [ ]:
@registry.node("greet", version=1, name="Greet", description="Greets a person")
def greet(
    name: Annotated[str, Text(label="Name")],
) -> Annotated[str, Output(label="Greeting")]:
    return f"Hi, {name}!"


nodes_c = [
    GraphNode("cond", "logic-if-equals@1", {"a": "Ada", "b": "Ada"}),
    GraphNode("hello", "greet@1", None),   # runs only if "Ada" == "Ada" — it does
    GraphNode("shout", "text-uppercase@1", None),
]
edges_c = [
    GraphEdge("e1", "cond", "hello", "output_1", "name"),
    GraphEdge("e2", "hello", "shout", "result", "text"),
]
compiled_c = compile(nodes=nodes_c, edges=edges_c, registry=registry)
print(await collect(execute(compiled_c)))